# DCGAN educativa para generación de rostros

Este cuaderno implementa y entrena una DCGAN sobre un subconjunto de LFW. Las imágenes se usan para generación sintética, no para identificar personas. Una ejecución breve sirve para observar el proceso, pero no garantiza rostros realistas. Ejecute en Colab con GPU.

In [ ]:
!pip -q install torch torchvision matplotlib


In [ ]:
import os, random, numpy as np, torch, matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import LFWPeople
from torchvision.utils import save_image, make_grid

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
size=64
tf=transforms.Compose([transforms.Resize(size), transforms.CenterCrop(size), transforms.ToTensor(), transforms.Normalize((.5,)*3,(.5,)*3)])
data=LFWPeople('/content/data', split='train', image_set='funneled', download=True, transform=tf)
data=Subset(data, list(range(min(2000,len(data)))))
loader=DataLoader(data,batch_size=64,shuffle=True,drop_last=True,num_workers=2)
print('Imágenes:',len(data),'Lotes:',len(loader))

In [ ]:
nz, ngf, ndf = 100, 64, 64
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.ConvTranspose2d(nz,ngf*8,4,1,0,bias=False),nn.BatchNorm2d(ngf*8),nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8,ngf*4,4,2,1,bias=False),nn.BatchNorm2d(ngf*4),nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4,ngf*2,4,2,1,bias=False),nn.BatchNorm2d(ngf*2),nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2,ngf,4,2,1,bias=False),nn.BatchNorm2d(ngf),nn.ReLU(True),
            nn.ConvTranspose2d(ngf,3,4,2,1,bias=False),nn.Tanh())
    def forward(self,z): return self.net(z)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,ndf,4,2,1,bias=False),nn.LeakyReLU(.2,True),
            nn.Conv2d(ndf,ndf*2,4,2,1,bias=False),nn.BatchNorm2d(ndf*2),nn.LeakyReLU(.2,True),
            nn.Conv2d(ndf*2,ndf*4,4,2,1,bias=False),nn.BatchNorm2d(ndf*4),nn.LeakyReLU(.2,True),
            nn.Conv2d(ndf*4,ndf*8,4,2,1,bias=False),nn.BatchNorm2d(ndf*8),nn.LeakyReLU(.2,True),
            nn.Conv2d(ndf*8,1,4,1,0,bias=False),nn.Sigmoid())
    def forward(self,x): return self.net(x).view(-1)
G,D=Generator().to(device),Discriminator().to(device)
for m in list(G.modules())+list(D.modules()):
    if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d,nn.BatchNorm2d)): nn.init.normal_(m.weight,0,.02)
loss=nn.BCELoss(); optG=optim.Adam(G.parameters(),lr=.0002,betas=(.5,.999)); optD=optim.Adam(D.parameters(),lr=.0002,betas=(.5,.999))
fixed=torch.randn(64,nz,1,1,device=device)

In [ ]:
epochs=3
loss_g=[]; loss_d=[]; os.makedirs('/content/resultados_dcgan',exist_ok=True)
for epoch in range(epochs):
    sumg=sumd=0.0
    for real,_ in loader:
        real=real.to(device); b=real.size(0); D.zero_grad(set_to_none=True)
        y=torch.ones(b,device=device); err_real=loss(D(real),y); err_real.backward()
        z=torch.randn(b,nz,1,1,device=device); fake=G(z)
        err_fake=loss(D(fake.detach()),torch.zeros(b,device=device)); err_fake.backward(); errd=err_real+err_fake; optD.step()
        G.zero_grad(set_to_none=True); errg=loss(D(fake),torch.ones(b,device=device)); errg.backward(); optG.step()
        sumg+=errg.item(); sumd+=errd.item()
    loss_g.append(sumg/len(loader)); loss_d.append(sumd/len(loader))
    with torch.no_grad(): sample=G(fixed).cpu()
    save_image(sample,f'/content/resultados_dcgan/epoca_{epoch+1}.png',normalize=True,nrow=8)
    print(f'Época {epoch+1}: D={loss_d[-1]:.3f}, G={loss_g[-1]:.3f}')
plt.plot(loss_d,label='Discriminador'); plt.plot(loss_g,label='Generador'); plt.xlabel('Época'); plt.ylabel('Pérdida'); plt.legend(); plt.grid(alpha=.2); plt.savefig('/content/resultados_dcgan/perdidas.png'); plt.show()
torch.save({'G':G.state_dict(),'D':D.state_dict(),'loss_g':loss_g,'loss_d':loss_d},'/content/resultados_dcgan/dcgan.pt')
print('Evidencias guardadas en /content/resultados_dcgan')

## Lectura de resultados

Compare las cuadrículas guardadas por época y la evolución de las pérdidas. Una ejecución corta suele producir imágenes borrosas; las pérdidas adversariales oscilan y no son por sí solas una medida de fidelidad o diversidad. Registre la configuración, número real de imágenes y las limitaciones observadas antes de redactar conclusiones.